Setup and Load Data: Loading the dataset and making a copy so we don't accidentally overwrite the original file.

In [10]:
import pandas as pd
import numpy as np

# Load the dirty data
df = pd.read_csv('dirty_dataset.csv')

# copy of dataset
df_clean = df.copy()

Handle Duplicates: Removing duplicate employee records to ensure each row represents a unique person.

In [11]:
# Drop rows that are exact duplicates across all columns
df_clean = df_clean.drop_duplicates()

# Drop rows that have the same employee_id, keeping only the first one
df_clean = df_clean.drop_duplicates(subset='employee_id', keep='first')

Age column contains missing values and impossible numbers like -5 and 200. Convert invalid ages to empty values (NaN) and then fill all missing values with the median age.

In [12]:
df_clean['age'] = pd.to_numeric(df_clean['age'], errors='coerce')

# Turn invalid ages into NaN (assuming normal working age is 18 to 70)
df_clean.loc[(df_clean['age'] < 18) | (df_clean['age'] > 80), 'age'] = np.nan

# Fill NaN with the median
median_age = df_clean['age'].median()
df_clean['age'] = df_clean['age'].fillna(median_age)

Salary column has dollar signs, commas, negative numbers, and extreme outliers. Remove text characters, convert everything to floats, fix the negatives, and replace massive outliers with the median salary.

In [13]:
# Remove dollar signs and commas
df_clean['salary'] = df_clean['salary'].astype(str).str.replace('$', '').str.replace(',', '')
df_clean['salary'] = pd.to_numeric(df_clean['salary'], errors='coerce')

# Turn negative salaries into positive ones
df_clean['salary'] = df_clean['salary'].abs()

# Find a normal median salary (ignoring the 9999999 outliers)
normal_median_salary = df_clean.loc[df_clean['salary'] < 1000000, 'salary'].median()

# Replace outliers and missing values with the normal median
df_clean.loc[df_clean['salary'] > 1000000, 'salary'] = normal_median_salary
df_clean['salary'] = df_clean['salary'].fillna(normal_median_salary)

Fixing the join_date column so all the different date formats are standardized into a single datetime format.

In [14]:
# Pandas automatically parses most mixed date formats
df_clean['join_date'] = pd.to_datetime(df_clean['join_date'], errors='coerce')

Standardizing text columns by making everything lowercase and fixing typos using the replace function.

In [15]:
# Clean Department
df_clean['department'] = df_clean['department'].astype(str).str.lower().str.strip()
df_clean['department'] = df_clean['department'].replace({
    'mrketing': 'marketing',
    'finace': 'finance'
})
df_clean['department'] = df_clean['department'].replace('nan', 'unknown')

# Clean Gender
df_clean['gender'] = df_clean['gender'].astype(str).str.lower().str.strip()
df_clean['gender'] = df_clean['gender'].replace({
    'm': 'male', 'man': 'male',
    'f': 'female', 'woman': 'female',
    'nan': 'unknown'
})

# Clean Country
df_clean['country'] = df_clean['country'].astype(str).str.lower().str.strip()
df_clean['country'] = df_clean['country'].replace({
    'america': 'usa', 'u.s.a': 'usa', 'us': 'usa',
    'u.k.': 'uk', 'united kingdom': 'uk',
    'bd': 'bangladesh', 'nan': 'unknown'
})

# Clean City
df_clean['city'] = df_clean['city'].astype(str).str.lower().str.strip()
df_clean['city'] = df_clean['city'].replace({
    'dhakka': 'dhaka', 'dhka': 'dhaka',
    'chittagongg': 'chittagong', 'chitagong': 'chittagong',
    'slyhet': 'sylhet', 'rajshai': 'rajshahi', 'nan': 'unknown'
})

Removing the "kg" text from the weight column to make it purely numeric, and standardizing the is_active column to just True/False values.

In [16]:
# Clean weight_kg
df_clean['weight_kg'] = df_clean['weight_kg'].astype(str).str.replace('kg', '', case=False)
df_clean['weight_kg'] = pd.to_numeric(df_clean['weight_kg'], errors='coerce')
df_clean['weight_kg'] = df_clean['weight_kg'].fillna(df_clean['weight_kg'].mean())

# Clean is_active
df_clean['is_active'] = df_clean['is_active'].astype(str).str.upper().str.strip()
df_clean['is_active'] = df_clean['is_active'].replace({
    '1': 'TRUE', '0': 'FALSE', 'NAN': 'UNKNOWN'
})

Fixing negative prices, filling in empty reviews, and dropping the completely redundant duplicate weight column.

In [17]:
# Clean price
df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce')
df_clean['price'] = df_clean['price'].abs() # fix negatives
df_clean['price'] = df_clean['price'].fillna(df_clean['price'].median())

# Clean review (fill missing)
df_clean['review'] = df_clean['review'].astype(str).replace('nan', 'No review')

# Drop the duplicate column
if 'weight_kg_duplic' in df_clean.columns:
    df_clean = df_clean.drop(columns=['weight_kg_duplic'])

# Make sure target is numeric
df_clean['target'] = pd.to_numeric(df_clean['target'], errors='coerce').fillna(0)

Verifying the final cleaned dataset and exporting it to a new CSV file.

In [18]:
# Check final info to make sure everything is non-null and right data type. After verification we save it.
print(df_clean.info())

df_clean.to_csv('cleaned_dataset.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 19801 entries, 0 to 20299
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   employee_id          19801 non-null  int64         
 1   name                 19801 non-null  object        
 2   age                  19801 non-null  float64       
 3   salary               19801 non-null  float64       
 4   join_date            16529 non-null  datetime64[ns]
 5   department           19801 non-null  object        
 6   gender               19801 non-null  object        
 7   country              19801 non-null  object        
 8   city                 19801 non-null  object        
 9   weight_kg            19801 non-null  float64       
 10  is_active            19801 non-null  object        
 11  review               19801 non-null  object        
 12  price                19801 non-null  float64       
 13  weight_kg_duplicate  0 non-null     